# OSMI Mental Health in Tech 2016 — Unsupervised Learning

This notebook documents the reproducible analytical workflow. The executable scripts remain the source of truth for the final results. The notebook provides an auditable narrative: data loading → feature engineering → clustering → evaluation → interpretation.

In [ ]:
from pathlib import Path
import pandas as pd
import sys
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
from src.data_loader import load_osmi_data
from src.feature_engineering import build_universal_core_matrix, build_workplace_support_matrix
df = load_osmi_data()
print(df.shape)


## 1. Primary respondent segmentation

The primary representation excludes employer-routing variables that could create structural missingness. Categorical responses are one-hot encoded with an explicit missing category; implausible ages outside 18–80 are treated as missing, median-imputed, and standardized.

In [ ]:
frame, X_df, encoder, variables = build_universal_core_matrix(df)
print('Respondents:', len(df))
print('Variables:', len(variables))
print('Encoded features:', X_df.shape[1])
print('\n'.join(variables))


In [ ]:
from src.clustering import kmeans_stability, fit_final_kmeans, evaluate_labels
X = X_df.to_numpy()
sweep, _ = kmeans_stability(X, k_values=range(2,9), seeds=range(10), n_init=50)
display(sweep)
model, labels = fit_final_kmeans(X, k=2, random_state=42, n_init=50)
print(evaluate_labels(X, labels))


In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=20).fit(X)
pca_table = pd.DataFrame({'PC': range(1,21), 'Variance': pca.explained_variance_ratio_, 'Cumulative': pca.explained_variance_ratio_.cumsum()})
display(pca_table.head(10))


## 2. Workplace-support segmentation

A secondary analysis is restricted to non-self-employed respondents (n=1,146) and focuses on current workplace support/stigma variables. Previous-employer variables, self-employment routing, and employer-type classification are excluded from the final representation to reduce structural confounding.

In [ ]:
subset, WX_df, wencoder, wvariables = build_workplace_support_matrix(df)
print('Respondents:', len(subset))
print('Variables:', len(wvariables))
print('Encoded features:', WX_df.shape[1])
print('\n'.join(wvariables))


In [ ]:
WX = WX_df.to_numpy()
w_sweep, _ = kmeans_stability(WX, k_values=range(2,9), seeds=range(10), n_init=50)
display(w_sweep)


## Interpretation principle

Internal metrics are treated as evidence rather than proof of naturally occurring employee types. Cluster interpretation is based on response profiles, stability, sensitivity to representation choices, and organizational actionability. The primary model is therefore not labelled as a diagnostic classifier.